In [27]:
import os
os.environ["GOOGLE_API_KEY"] = ""

##Install libraries

In [18]:
!pip install langchain faiss-cpu  google-generativeai langchain-google-genai tiktoken  langchain-community wikipedia youtube-transcript-api  \
                python-dotenv

In [19]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

##Step 1a - Indexing (Document Ingestion)

In [20]:
video_id = "Gfr50f6ZBvo" # only the ID, not full URL
try:
    fetched_transcript = YouTubeTranscriptApi().fetch(video_id, languages=['en'])
    transcript_list = fetched_transcript.to_raw_data()
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"An error occurred: {e}")

the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough 

In [21]:
transcript_list

[{'text': 'the following is a conversation with',
  'start': 0.08,
  'duration': 3.44},
 {'text': 'demus hasabis', 'start': 1.76, 'duration': 4.96},
 {'text': 'ceo and co-founder of deepmind', 'start': 3.52, 'duration': 5.119},
 {'text': 'a company that has published and builds',
  'start': 6.72,
  'duration': 4.48},
 {'text': 'some of the most incredible artificial',
  'start': 8.639,
  'duration': 4.561},
 {'text': 'intelligence systems in the history of',
  'start': 11.2,
  'duration': 4.8},
 {'text': 'computing including alfred zero that',
  'start': 13.2,
  'duration': 3.68},
 {'text': 'learned', 'start': 16.0, 'duration': 2.96},
 {'text': 'all by itself to play the game of gold',
  'start': 16.88,
  'duration': 4.559},
 {'text': 'better than any human in the world and',
  'start': 18.96,
  'duration': 5.6},
 {'text': 'alpha fold two that solved protein',
  'start': 21.439,
  'duration': 4.241},
 {'text': 'folding', 'start': 24.56, 'duration': 4.16},
 {'text': 'both tasks consider

In [22]:
transcript

"the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough

##Step 1b - Indexing (Text Splitting)

In [23]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [24]:
len(chunks)

168

In [25]:
chunks[100]

Document(metadata={}, page_content="and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask w

##Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [30]:
!pip install -U sentence-transformers langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 15.2 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0


In [31]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [32]:
text = "This is a YouTube video about artificial intelligence."

vector = embedding_model.embed_query(text)

print("Embedding dimensions:", len(vector))
print(vector[:5])

Embedding dimensions: 384
[-0.023492274805903435, -0.043512411415576935, 0.0012622503563761711, -0.015527399256825447, 0.04441428184509277]


In [33]:

vector_store = FAISS.from_documents(chunks, embedding_model)

In [34]:
vector_store.index_to_docstore_id

{0: '256d89b0-8874-45f5-9386-42f0415f7b50',
 1: 'e7460782-bdc4-4a49-bfcf-d4982cdadd30',
 2: '394335b4-c47f-4b96-a1c0-5f4b17a37ed7',
 3: 'ddfd28d5-5317-4a00-8e54-cd7aa9fde6de',
 4: 'f71644bc-f188-4d94-93ad-e9c7e5f26ca6',
 5: '408f0687-c4d6-4533-a23e-97b4214254c5',
 6: '089ffcb8-759e-4bf1-893e-9414344a960c',
 7: 'c05247e7-80ef-4315-bf65-a7e7383f90a3',
 8: 'cd99286d-c79c-42c8-ab7d-1af4f69b637e',
 9: '4d2c2094-be53-4f65-897b-b23d6174b5dd',
 10: '080594d5-30e5-4dc2-8852-e4e52996e2ab',
 11: '14bbe5d7-779d-491e-a9be-dbaa8cde0532',
 12: 'de004dd1-7622-4f26-8031-459bb7bcf6a1',
 13: '92937c2d-f4da-4500-8ada-a0fd3dc0e7fb',
 14: '181c2be7-7d62-46ef-a14a-652737b12631',
 15: 'b6ebbfc8-d48d-4467-8625-289b85842b34',
 16: '0fd48bb7-7a34-4f99-886e-e54cb7fd13be',
 17: '98e7e335-79d9-4b31-a265-e35e648afed4',
 18: 'fb3e2886-d6ec-4c23-8f31-ba71bc29b967',
 19: '805b1477-9003-44e9-9961-b734be5562fd',
 20: 'ac70ebe0-d8ae-41e6-8832-49968baf95b8',
 21: '7e552355-4e22-495a-b651-cbaa8c3926be',
 22: '12dcc958-f5b6-

In [35]:
vector_store.get_by_ids(['b971573a-be48-4e24-87e4-0c9f607bf214'])

[Document(id='b971573a-be48-4e24-87e4-0c9f607bf214', metadata={}, page_content='demas establish to support this podcast please check out our sponsors in the description and now let me leave you with some words from edskar dykstra computer science is no more about computers than astronomy is about telescopes thank you for listening and hope to see you next time')]

##Step 2 - Retrieval

In [36]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [37]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7ca8585bde80>, search_kwargs={'k': 4})

In [38]:
retriever.invoke('What is deepmind')

[Document(id='51d51bcf-206f-43ea-a0ab-8f0a56d69ca8', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

##Step 3 - Augmentation

In [40]:
llm = ChatGoogleGenerativeAI(model = 'gemini-3.8-flash', temperature=0.2)

In [41]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [42]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [43]:
retrieved_docs

[Document(id='d3967f37-1771-4fde-9b4e-77e9376d5d95', metadata={}, page_content="in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be intere

In [44]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be interesting from a research perspective from our point of view from an ai point of\n\

In [45]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [46]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we 

##Step 4 - Generation

In [47]:
answer = llm.invoke(final_prompt)
print(answer.content)

[{'type': 'text', 'text': 'Yes, nuclear fusion is discussed. Based on the transcript, the following points were discussed:\n\n* **Role of AI in Energy and Climate:** Nuclear fusion is identified as an energy and climate challenge where AI can help accelerate progress.\n* **Challenges in Fusion:** Fusion involves significant challenges in physics, material science, and engineering to build massive reactors and contain high-temperature plasma.\n* **Approach to Solving Bottlenecks:** The team works from first principles with domain experts to identify the bottlenecks preventing fusion from working today, focusing on those amenable to AI methods.\n* **Collaboration with EPFL:** They collaborated with the Swiss Technical Institute (EPFL) in Switzerland, using their test reactor to run experiments.\n* **Deep Reinforcement Learning for Plasma Control:** They published a *Nature* paper on the magnetic control of tokamak plasmas using deep reinforcement learning. \n* **Results and Containment:*

##Building a Chain

In [49]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [50]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [51]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [52]:
parser = StrOutputParser()

In [53]:
main_chain = parallel_chain | prompt | llm | parser

In [54]:
main_chain.invoke('Can you summarize the video')

"Based on the provided transcript, the conversation covers the following topics:\n\n* **Explaining Complex Concepts & Physics:** The speakers discuss how advanced intelligence or AI might begin explaining reality through deeper, simpler fundamental physics beyond the standard model, addressing long-standing mysteries like consciousness, life, and gravity.\n* **Intelligence and Simplicity:** Referencing Richard Feynman, they note that a true sign of intelligence is the ability to explain complex topics simply and clearly. They also touch upon how humans already have a symbiotic relationship with computing devices.\n* **Chess and Computing History:** A speaker reflects on early computing history, including Claude Shannon's 1949 chess paper and Alan Turing manually running his own chess program on paper. They also recall Deep Blue's match against Garry Kasparov, noting they were ultimately more impressed by Kasparov’s versatile human mind than the brute-force calculation machine. \n* **Cl